# Tiki Kitchen Appliances Market Analytics
## Data Cleaning

This notebook cleans the raw datasets based on the findings documented in
`data_check_report.md` (output of `data_quality_check.ipynb`).

Every transformation below is tied to a specific finding from the audit —
this notebook does not introduce new cleaning decisions that were not
already surfaced and reasoned about during the quality check.

### Steps
1. Setup
2. Load Data
3. Drop Unused Columns
4. Deduplication (defensive re-check)
5. Standardize Text Fields
6. Handle Missing Values by Business Meaning
7. Validate Price / Discount Logic (regression check)
8. Create Analytical Segments
9. Reconcile Review Counts
10. Save Cleaned Data
11. Cleaning Summary


### 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)


### 2. Load Data

In [2]:
RAW_PRODUCTS = "../data/raw/crawled_product_data.csv"
RAW_REVIEWS = "../data/raw/comments_data.csv"

OUT_PRODUCTS = "../data/processed/products_cleaned.csv"
OUT_REVIEWS = "../data/processed/reviews_cleaned.csv"

products = pd.read_csv(RAW_PRODUCTS)
reviews = pd.read_csv(RAW_REVIEWS)

n_products_before = len(products)
n_reviews_before = len(reviews)

print("Products:", products.shape)
print("Reviews:", reviews.shape)


Products: (2001, 30)
Reviews: (48649, 10)


### 3. Drop Unused Columns

`meta_title` was found to be 100% missing (0/2,001 non-null) during the
data quality audit. It carries no analytical value and is dropped here.

In [3]:
cols_to_drop = ["meta_title"]

products = products.drop(
    columns=[c for c in cols_to_drop if c in products.columns]
)

print("Dropped columns:", cols_to_drop)
print("Remaining columns:", products.shape[1])


Dropped columns: ['meta_title']
Remaining columns: 29


### 4. Deduplication (defensive re-check)

The audit reported 0 duplicate `id` / `comment_id` values. This step is
kept anyway as a defensive check — cleaning code should not blindly trust
a prior audit run against a possibly different raw file snapshot.

In [4]:
before_p = len(products)
products = products.drop_duplicates(subset=["id"])
print(f"Products deduped: {before_p} -> {len(products)} "
      f"(removed {before_p - len(products)})")

before_r = len(reviews)
reviews = reviews.drop_duplicates(subset=["comment_id"])
print(f"Reviews deduped: {before_r} -> {len(reviews)} "
      f"(removed {before_r - len(reviews)})")


Products deduped: 2001 -> 2001 (removed 0)
Reviews deduped: 48649 -> 48649 (removed 0)


### 5. Standardize Text Fields

Trim whitespace and normalize `NaN` representation for text fields that
will later be used as groupby keys (`brand_name`, `seller_name`,
`category_name`).

In [5]:
text_cols = ["brand_name", "seller_name", "category_name"]

for col in text_cols:
    if col in products.columns:
        products[col] = products[col].astype(str).str.strip()
        products[col] = products[col].replace({"nan": np.nan})

products[text_cols].head()


,brand_name,seller_name,category_name
0,Bluestone,Tiki Trading,Điện Gia Dụng
1,Sharp,Tiki Trading,CHƯƠNG TRÌNH KHUYẾN MẠI KHÁC
2,Sharp,Tiki Trading,Điện Gia Dụng
3,Cuckoo,Tiki Trading,Điện Gia Dụng
4,Philips,Tiki Trading,Máy ép nhanh


### 6. Handle Missing Values by Business Meaning

Two missing-value patterns were identified in the audit, each with a
different correct treatment:

- **`quantity_sold_value`** missing (~42.5% of rows) does not mean "sold
  0 units" — it means no sales have ever been recorded for that product.
  Blindly filling with 0 would erase this distinction. A `has_sold` flag
  is created instead, and a filled column is kept separately for
  aggregation convenience.
- **`seller_id` / `category_id`** missing (8 records) likely reflects
  products that became unavailable/delisted between crawl steps. Per the
  audit's Action Plan, these are flagged for manual review rather than
  dropped outright.

In [6]:
products["has_sold"] = products["quantity_sold_value"].notna()
products["quantity_sold_value_filled"] = products["quantity_sold_value"].fillna(0)

print("has_sold = True :", products["has_sold"].sum())
print("has_sold = False:", (~products["has_sold"]).sum())


has_sold = True : 1150
has_sold = False: 851


In [7]:
products["possibly_delisted"] = (
    products["seller_id"].isna() | products["category_id"].isna()
)

print("possibly_delisted flagged:", products["possibly_delisted"].sum())
products.loc[products["possibly_delisted"], ["id", "product_name", "seller_id", "category_id"]]


possibly_delisted flagged: 8


,id,product_name,seller_id,category_id
234,49601294,Nồi cơm điện Tiger JNP-1000 (Màu trắng) - Hàng...,NaN,NaN
418,127031344,Máy Nướng Bánh Mì Kẹp BlueStone SBB-2333 (650W...,NaN,NaN
479,147903536,Bếp Từ Đơn Sunhouse SHD6803 (2000W) - Kèm Nồ...,NaN,NaN
728,205818604,[HÀNG CHÍNH HÃNG] Máy Ép Chậm Hurom H100S,NaN,NaN
1521,276849824,NỒI CHIÊN KHÔNG DẦU 9L SUNHOUSE SHD4037 - Hàng...,NaN,NaN
1600,277512215,Nồi cơm điện tử Kangaroo 1.5 lít KG15RCE2 - Hà...,NaN,NaN
1678,278096977,Máy xay sinh tố Locknlock 950ml Duo Turbo Blen...,NaN,NaN
1888,278922364,Máy pha viên nén Trà và Cafe Wells Home Cafe [...,NaN,NaN


### 7. Validate Price / Discount Logic (regression check)

`original_price` is used as the reference price (more reliable than
`list_price` in general, though the audit confirmed `list_price` is also
100% valid in this particular dataset). `discount_rate` is recomputed
independently and compared to the crawled value to catch inconsistencies.

The two `assert` checks re-verify that the validity findings from the
audit (0 invalid prices, 0 `original_price < price` logic errors) still
hold after the transformations above — cleaning code should not silently
break what was already confirmed clean.

In [8]:
products["reference_price"] = products["original_price"].where(
    products["original_price"] > 0, products["price"]
)

products["discount_rate_calc"] = (
    (products["reference_price"] - products["price"])
    / products["reference_price"] * 100
).round(1)

products["discount_rate_mismatch"] = (
    (products["discount_rate"] - products["discount_rate_calc"]).abs() > 5
)

n_mismatch = products["discount_rate_mismatch"].sum()
print(f"discount_rate mismatch (>5pp vs recalculated): {n_mismatch} "
      f"({n_mismatch / len(products) * 100:.1f}%)")


discount_rate mismatch (>5pp vs recalculated): 1 (0.0%)


In [9]:
assert (products["price"] <= 0).sum() == 0, "Found price <= 0 after cleaning!"
assert (products["original_price"] < products["price"]).sum() == 0, (
    "Found original_price < price after cleaning!"
)

print("Regression check passed: price / original_price logic still valid.")


Regression check passed: price / original_price logic still valid.


### 8. Create Analytical Segments

`price_bucket` and `discount_bucket` are created here so that EDA and
KPI notebooks downstream can reuse a single, consistent segmentation
instead of redefining bins in multiple places.

In [10]:
products["price_bucket"] = pd.cut(
    products["price"],
    bins=[0, 100_000, 300_000, 700_000, 1_500_000, float("inf")],
    labels=["<100k", "100-300k", "300-700k", "700k-1.5tr", ">1.5tr"],
)

products["discount_bucket"] = pd.cut(
    products["discount_rate"],
    bins=[-0.1, 0, 10, 20, 30, 50, 100],
    labels=["0%", "0-10%", "10-20%", "20-30%", "30-50%", ">50%"],
)

products[["price", "price_bucket", "discount_rate", "discount_bucket"]].head()


,price,price_bucket,discount_rate,discount_bucket
0,551000,300-700k,34,30-50%
1,756000,700k-1.5tr,20,10-20%
2,2061000,>1.5tr,21,20-30%
3,1490000,700k-1.5tr,35,30-50%
4,932000,700k-1.5tr,36,30-50%


### 9. Reconcile Review Counts

The audit found that `review_count` (a static field on the product page)
can diverge substantially from the actual number of reviews collected by
the crawler — most noticeably for product `id 392842` (declared: 3,
crawled: 170). The crawled count is treated as ground truth, since it
reflects data measured directly rather than a potentially stale page
field.

`crawled_review_count` is computed here and merged into `products`;
`review_count` is kept as-is (not overwritten) so both can be compared
later if needed.

In [11]:
crawled_review_counts = (
    reviews.groupby("product_id").size().reset_index(name="crawled_review_count")
)

products = products.merge(
    crawled_review_counts, left_on="id", right_on="product_id", how="left"
)
products["crawled_review_count"] = products["crawled_review_count"].fillna(0).astype(int)
products = products.drop(columns=["product_id"])

products[["id", "review_count", "crawled_review_count"]].head()


,id,review_count,crawled_review_count
0,359479,1232,1245
1,368305,40,48
2,368309,47,59
3,374109,10,17
4,392729,371,400


In [12]:
large_mismatch = products[
    (products["review_count"] > 0)
    & (products["crawled_review_count"] > 0)
    & (
        (products["crawled_review_count"] / products["review_count"].replace(0, np.nan))
        > 3
    )
]

print(f"Products with crawled_review_count > 3x declared review_count: {len(large_mismatch)}")
large_mismatch[["id", "product_name", "review_count", "crawled_review_count"]]


Products with crawled_review_count > 3x declared review_count: 53


,id,product_name,review_count,crawled_review_count
6,392842,Bình Đun Siêu Tốc Philips HD9306 (1.5L) - Hàng...,3,170
12,403444,Máy Pha Cà Phê Espresso Tiross TS620 - Hàng Ch...,1,30
18,419160,Máy Xay Thịt Bosch MMR08R2 - Hàng chính hãng,1,28
26,452600,Máy Xay Cầm Tay Panasonic PASO-MX-GS1WRA - Hàn...,1,32
46,548599,Máy Pha Cà Phê Espresso Tiross TS-621 (4 bar) ...,6,35
60,557855,Nồi Áp Suất Đa Năng Sunhouse DNDSHD1552 - 5L (...,2,14
65,558436,Lò Nướng Thủy Tinh Tiger Queen AX-777MV - 11L ...,1,8
74,563737,Nồi Cơm Điện Tử Lock&Lock EJR351BRW (1.8 Lít) ...,3,30
87,792752,Ấm Siêu Tốc Trường Thọ BA 2088 - Xanh (5L)- Hã...,1,5
97,1460199,Bếp Nướng Điện Sunhouse SHD4607 (1500W) - Hàng...,2,30


### 10. Save Cleaned Data

In [13]:
os.makedirs(os.path.dirname(OUT_PRODUCTS), exist_ok=True)

products.to_csv(OUT_PRODUCTS, index=False, encoding="utf-8-sig")
reviews.to_csv(OUT_REVIEWS, index=False, encoding="utf-8-sig")

print(f"Saved cleaned products -> {OUT_PRODUCTS} ({len(products)} rows)")
print(f"Saved cleaned reviews  -> {OUT_REVIEWS} ({len(reviews)} rows)")


Saved cleaned products -> ../data/processed/products_cleaned.csv (2001 rows)
Saved cleaned reviews  -> ../data/processed/reviews_cleaned.csv (48649 rows)


### 11. Cleaning Summary

In [14]:
summary = pd.DataFrame([
    {"metric": "Products (before -> after)",
     "value": f"{n_products_before} -> {len(products)}"},
    {"metric": "Reviews (before -> after)",
     "value": f"{n_reviews_before} -> {len(reviews)}"},
    {"metric": "Columns dropped",
     "value": ", ".join(cols_to_drop)},
    {"metric": "possibly_delisted flagged",
     "value": int(products["possibly_delisted"].sum())},
    {"metric": "discount_rate_mismatch flagged",
     "value": int(products["discount_rate_mismatch"].sum())},
    {"metric": "Large review_count mismatch (>3x)",
     "value": len(large_mismatch)},
])

display(summary)


,metric,value
0,Products (before -> after),2001 -> 2001
1,Reviews (before -> after),48649 -> 48649
2,Columns dropped,meta_title
3,possibly_delisted flagged,8
4,discount_rate_mismatch flagged,1
5,Large review_count mismatch (>3x),53


In [15]:
cleaning_report_lines = [
    "# Data Cleaning Report",
    "",
    "## Summary",
    "",
] + [f"- **{row['metric']}**: {row['value']}" for _, row in summary.iterrows()] + [
    "",
    "## Products Flagged for Manual Review",
    "",
    f"- `possibly_delisted = True` IDs: "
    f"{products.loc[products['possibly_delisted'], 'id'].tolist()}",
    f"- Large review_count mismatch (>3x) IDs: "
    f"{large_mismatch['id'].tolist()}",
]

with open("cleaning_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaning_report_lines))

print("Saved cleaning_report.md")


Saved cleaning_report.md
